In [52]:
import numpy as np
import matplotlib.pyplot as plt

# Vertex shading

---
# Load inputs

## OBJ & PLY

In [53]:
class Mesh:
    def __init__(self):
        self.vertices = np.array([[]])
        self.faces = np.array([[]])
        self.colors = np.array([[]])

        #calculated results
        self.clip_space_vertices = np.array([[]])
        self.NDC_vertices = np.array([[]])
        self.screen_space_vertices = np.array([[]])

    def load_obj(self, filepath):
        """
        Load a .obj file and extract vertex positions.
        filepath: Path to the .obj file
        Returns: Nx3 numpy array of vertex positions
        """
        vertices = []
        faces = []
        with open(filepath, 'r') as file:
            for line in file:
                if line.startswith('v '):  # Line defines a vertex
                    parts = line.strip().split()
                    if len(parts) >= 4:
                        vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
                elif line.startswith('f '):  # Line defines a face
                    parts = line.strip().split()
                    # OBJ indices are 1-based, so we subtract 1 for 0-based indexing
                    face = [int(part.split('/')[0]) - 1 for part in parts[1:4]]
                    faces.append(face)
        self.vertices = np.array(vertices)
        self.faces = np.array(faces)

    def load_ply(self, filepath):
        """
        Load a .ply file and extract vertex positions and face indices.
        Assumes the PLY file is in ASCII format with properties: x, y, z, red, green, blue
        and faces defined as a list of vertex indices.

        filepath: Path to the .ply file
        """
        with open(filepath, 'r') as file:
            line = file.readline().strip()
            if line != 'ply':
                raise ValueError("The file does not start with 'ply' header.")

            # Initialize variables
            num_vertices = 0
            num_faces = 0
            header_ended = False
            properties = []

            # Parse header
            while not header_ended:
                line = file.readline().strip()
                if line.startswith('element vertex'):
                    num_vertices = int(line.split()[-1])
                elif line.startswith('element face'):
                    num_faces = int(line.split()[-1])
                elif line.startswith('property'):
                    properties.append(line)
                elif line == 'end_header':
                    header_ended = True

            # Determine property indices (optional: if you want to store colors)
            property_names = [prop.split()[-1] for prop in properties]
            try:
                x_idx = property_names.index('x')
                y_idx = property_names.index('y')
                z_idx = property_names.index('z')
            except ValueError:
                raise ValueError("PLY file does not contain x, y, z properties.")

            # Optional: Check for color properties
            has_color = all(color in property_names for color in ['red', 'green', 'blue'])
            if has_color:
                red_idx = property_names.index('red')
                green_idx = property_names.index('green')
                blue_idx = property_names.index('blue')

            # Read vertex data
            vertices = []
            colors = []
            for _ in range(num_vertices):
                parts = file.readline().strip().split()
                if len(parts) < 3:
                    raise ValueError("Vertex line does not have enough coordinates.")
                vertex = [float(parts[x_idx]), float(parts[y_idx]), float(parts[z_idx])]
                vertices.append(vertex)
                if has_color:
                    color = [int(parts[red_idx]), int(parts[green_idx]), int(parts[blue_idx])]
                    colors.append(color)

            self.vertices = np.array(vertices)
            if has_color:
                self.colors = np.array(colors)
            else:
                self.colors = None  # Or handle as needed

            # Read face data
            faces = []
            for _ in range(num_faces):
                parts = file.readline().strip().split()
                if len(parts) < 4:
                    raise ValueError("Face line does not have enough indices.")
                vertex_count = int(parts[0])
                if vertex_count != 3:
                    raise ValueError("Only triangular faces are supported.")
                # PLY indices are 0-based
                face = [int(idx) for idx in parts[1:4]]
                faces.append(face)

            self.faces = np.array(faces)

## Camera

**Arguments**  
1. Camera postions
  - `eye`: The position of the camara.
  - `center`: The center of the screen space.  
    - Camera facing direction: (`eye` - `center`)
  - `up`: The Up direction of the camera.  
2. Screen
  - `screen_W`: The width of the screen.
  - `screen_H`: The height of the screen.
    - aspect_ratio: `screen_W` / `screen_H`
  - `screen_buffer`: The array with W*H pixels, each presented with RGB.
3. Projection parameters
  - `near_clipping_plane`: The near clipping plane.
  - `far_clipping_plane`: The far clipping plane.
  - `fov`: The vertival field of view in degrees.(FOVy)

In [54]:
class Camera:
    def __init__(self, eyeX: float,   eyeY: float,   eyeZ: float,
              centerX: float,  centerY: float, centerZ: float,
              upX: float,    upY: float,   upZ: float,
              screen_W: int, screen_H: int,
              near_clipping_plane: float, far_clipping_plane: float,
              fov: float):
        self.eyeX = eyeX
        self.eyeY = eyeY
        self.eyeZ = eyeZ
        self.eye = np.array([eyeX, eyeY, eyeZ])
        self.centerX = centerX
        self.centerY = centerY
        self.centerZ = centerZ
        self.center = np.array([centerX, centerY, centerZ])
        self.upX = upX
        self.upY = upY
        self.upZ = upZ
        self.up = np.array([upX, upY, upZ])

        self.screen_W = screen_W
        self.screen_H = screen_H
        self.aspect_ratio = screen_W / screen_H
        self.screen_buffer = np.zeros((screen_H, screen_W, 3), dtype=np.uint8)
        self.screen_buffer.fill(255)
        self.screen_depth_buffer = np.zeros((screen_H, screen_W))
        self.screen_depth_buffer.fill(1.1)

        self.near_clipping_plane = near_clipping_plane
        self.far_clipping_plane = far_clipping_plane
        self.fov = fov
    def set_eye(self, eyeX: float, eyeY: float, eyeZ: float):
        self.eyeX = eyeX
        self.eyeY = eyeY
        self.eyeZ = eyeZ
        self.eye = np.array([eyeX, eyeY, eyeZ])
    def draw_screen(self):
        plt.imshow(self.screen_buffer)
        plt.axis('off')
        plt.show()


In [55]:
camera = Camera( eyeX=4, eyeY=4, eyeZ=4,
          centerX=0, centerY=0, centerZ=0,
          upX=0, upY=1, upZ=0,
          screen_W=1280, screen_H=720,
          near_clipping_plane=0.1, far_clipping_plane=10,
          fov=45 )

# Vertex Shading

Project vertexes onto clip space by performong the MVP transformation.  

**References**  
- [OpenGL中投影矩阵(Projection Matrix)详解 - CSDN](https://blog.csdn.net/qq_39300235/article/details/90670282)
- [OpenGL矩阵变换的数学推导 - 騰訊雲](https://cloud.tencent.com/developer/article/1389550)
- [Computing FOVX (openGL) - stackoverflow](https://stackoverflow.com/questions/5504635/computing-fovx-opengl)

## Model Matrix

_**SKIP FIRST**_  
We directly put the mesh in the center.

Local space -> Model space

In [56]:
def calc_Model_matrix( camera: Camera, mesh: Mesh):
  Model = np.eye(4)
  return Model

## View Matrix

Model space -> View space

The coordinates of the camera is as follows:
- `z axis`: The camera facing direction. ( eye - center )
- `x axis`: The right direction of the camera. ( cross(z, u) )
- `y axis`: The up direction of the camera. ( cross(z, x) )

### Find inverse square

#### Newton-Raphson Method

This method can help us find the x, where f(x) = 0.
1. Initial X0 with LUT.
2. Find the intersection of the tangent line of f(x) on Xn and refresh Xn.

$$X_{n+1} = X{n} - \frac{f(X_n)}{f'(X_n)}$$

3. repeat step 2. to get a higher precision

---
We want to find $$x = \frac{1}{\sqrt{c}}$$

which is equivalent to finding $$f(x) = \frac{1}{x^2} - c = 0$$
THerefore, we can find Xn by Newton-Raphson method
$$X_{n+1} = X{n} - \frac{f(X_n)}{f'(X_n)} = x_n \cdot \frac{3 - c \cdot x_n^2}{2}$$


In [58]:
def LUT( C ):
  c = int(C)
  c = ((1/np.sqrt(c))*(2**21))//(2) / 2**20
  return c

In [59]:
def INV_SQRT( C ):
  c = (C + 2**19) // 2**20
  x = LUT( c )
  for i in range(3):
    # print(i, x)
    x = x * ( 3 - ( C/(2**20) ) * x * x ) / 2
  return x

In [60]:
def calc_View_matrix( camera: Camera):
    # Calculate the Z axis
    Z = camera.eye - camera.centerZ
    Z = Z / np.linalg.norm(Z)

    # Calculate the X axis
    X = np.cross(camera.up, Z)
    X = X / np.linalg.norm(X)

    # Calculate the Y axis
    Y = np.cross(Z, X)
    Y = Y / np.linalg.norm(Y)

    View = [ [ X[0], Y[0], Z[0], -np.dot(X, camera.eye)],
          [ X[1], Y[1], Z[1], -np.dot(Y, camera.eye)],
          [ X[2], Y[2], Z[2], -np.dot(Z, camera.eye)],
          [ 0.00, 0.00, 0.00,      1.00    ] ]
    View = np.array(View)
    return View

## Projection Matrix

View Space -> Clip Space

Perspective projection.
1. We need to translate FOVy from degree to radius.
2. Calculate fy and fx.
3. Derive the projection matrix.

In [62]:
def calc_Projection_matrix( camera: Camera):

  fy = 1.0 / np.tan( np.radians( camera.fov ) / 2 )
  fx = fy / camera.aspect_ratio

  far = camera.far_clipping_plane
  near = camera.near_clipping_plane
  Projection = [ [ fx,  0,       0       ,       0        ],
          [ 0,  fy,       0       ,       0        ],
          [ 0,  0, - (far+near) / (far-near), (-2*far*near) / (far-near) ],
          [ 0,  0,       -1       ,       0        ] ]

  Projection = np.array(Projection)
  return Projection

## Perform MVP Transformation

1. Expand the vertices into homogeneous matrix
2. Then multiply the matrixes

In [64]:
def perform_MVP_transformation( camera: Camera, mesh: Mesh):
  M = calc_Model_matrix(camera, mesh)
  V = calc_View_matrix(camera)
  P = calc_Projection_matrix(camera)
  MVP = P @ V @ M
  vertices_homogeneous = np.hstack((mesh.vertices, np.ones((mesh.vertices.shape[0], 1))))
  mesh.clip_space_vertices = vertices_homogeneous @ MVP.T
  return mesh

## NDC Transformation

Normalized Device Coordinate Transformation.  

Original vertices: [ Xc, Yc, Zc, Wc ]  
NDC vertices: [ Xc/Wc, Yc/Wc, Zc/Wc ]

In [66]:
def NDC_transformation(mesh):
  ndc_space_vertices = []

  for vertex in mesh.clip_space_vertices:
      x_c, y_c, z_c, w_c = vertex

      x_ndc = x_c / w_c
      y_ndc = y_c / w_c
      z_ndc = z_c / w_c

      ndc_space_vertices.append([x_ndc, y_ndc, z_ndc])

  mesh.NDC_vertices = np.array(ndc_space_vertices)
  return mesh

## NDC to Screen

**Note that the Y coordinate should be flipped**

- Upper left corner: (0,0)  
- Lower right corener: (width, height)  

In [68]:
def NDC_to_screen(camera: Camera, mesh: Mesh):
  ndc_x = mesh.NDC_vertices[:, 0]
  ndc_y = mesh.NDC_vertices[:, 1]

  screen_x = (ndc_x + 1) / 2 * camera.screen_W
  screen_y = (1 - (ndc_y + 1) / 2) * camera.screen_H

  mesh.screen_space_vertices = np.column_stack((screen_x, screen_y))
  return mesh

## Test the function

Output the function into GIF and see if it works well.

- `plot_screen_coords(screen_coords, width, height):`


In [70]:
import matplotlib.pyplot as plt

def plot_screen_coords(screen_coords, width, height):
    """
    Plots screen coordinates on a screen-sized canvas and returns the figure.

    Parameters:
    screen_coords (numpy array): Array of screen coordinates, shape (N, 2).
    width (int): Screen width in pixels.
    height (int): Screen height in pixels.

    Returns:
    matplotlib.figure.Figure: The figure object for further processing.
    """
    # Create a figure with a fixed aspect ratio to match screen dimensions
    fig, ax = plt.subplots(figsize=(width / 100, height / 100))  # Scaling by 100 for proportionality
    ax.scatter(screen_coords[:, 0], screen_coords[:, 1], color='red', label='Screen Coordinates')

    # Set screen limits
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)  # Invert Y-axis since screen Y-axis goes downward

    # Titles and labels
    ax.set_title("Screen Space Coordinates")
    ax.set_xlabel("X (pixels)")
    ax.set_ylabel("Y (pixels)")

    # Add grid and legend
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    return fig  # Return the figure object

In [74]:
import math
from tqdm import tqdm
import imageio
import numpy as np
import matplotlib.pyplot as plt

# Assuming the required classes and methods (Mesh, Camera, etc.) are defined elsewhere
mesh = Mesh()
mesh.load_obj('/content/drive/MyDrive/ICLAB_mesh/teapot.obj')

frames = []

# Initialize tqdm with dynamic postfix updates
with tqdm(total=(2 * 3 * 360) // (2 * 3), desc="Rendering frames", postfix={"Camera position": "(0, 0, 0)"}) as pbar:
    for rad in range(0, 2 * 3 * 360, 2 * 3):
        distance = 5
        rad /= 360
        CamX = distance * math.cos(rad)
        CamY = distance
        CamZ = distance * math.sin(rad)

        # Update the progress bar's postfix with the current camera position
        pbar.set_postfix({"Camera position": f"({CamX:.2f}, {CamY:.2f}, {CamZ:.2f})"})

        camera = Camera(
            eyeX=CamX, eyeY=CamY, eyeZ=CamZ,
            centerX=0, centerY=0, centerZ=0,
            upX=0, upY=1, upZ=0,
            screen_W=1280, screen_H=720,
            near_clipping_plane=0.1, far_clipping_plane=1000,
            fov=45
        )
        mesh = perform_MVP_transformation(camera, mesh)
        mesh = NDC_transformation(mesh)
        mesh = NDC_to_screen(camera, mesh)

        fig = plot_screen_coords(mesh.screen_space_vertices, camera.screen_W, camera.screen_H)

        # Save the figure as an image in memory
        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)

        plt.close(fig)  # Close the figure to free memory

        # Update the progress bar
        pbar.update(1)

# Save all frames into a .gif
output_path = "/content/drive/MyDrive/ICLAB_mesh/teapot.gif"
imageio.mimsave(output_path, frames, fps=24)

print(f"GIF saved to {output_path}")

# Rasterization

## Determine if the point is in triangle
```
    ( x1, y1 )
       /\
      /  \
     /    \
    -- -- --
( x2, y2 )   ( x3, y3 )
```


### Vector Cross Product Method (Half-Plane Technique)

**Concept:**  
This method uses the sign of the cross product to determine if the point lies on the same side of all the triangle's edges.

**Steps:**

1. **Define the Triangle and Point:**
   - Triangle vertices \( A(x_A, y_A) \), \( B(x_B, y_B) \), \( C(x_C, y_C) \).
   - Point \( P(x, y) \).

2. **Compute Vectors:**
   - Compute vectors for the edges \( AB \), \( BC \), and \( CA \).
   - Compute vectors from the triangle's vertices to point \( P \).

3. **Calculate Cross Products:**
   \begin{align*}
   \text{Cross}_1 &= (B_x - A_x)(P_y - A_y) - (B_y - A_y)(P_x - A_x) \\
   \text{Cross}_2 &= (C_x - B_x)(P_y - B_y) - (C_y - B_y)(P_x - B_x) \\
   \text{Cross}_3 &= (A_x - C_x)(P_y - C_y) - (A_y - C_y)(P_x - C_x)
   \end{align*}

4. **Check the Signs:**
   - If all cross products have the same sign (all positive or all negative), then \( P \) is inside the triangle.
   - If the signs differ, \( P \) is outside the triangle.


#### By Cross Product

In [77]:
def sign(p1, p2, p3):
    return (p1[0] - p3[0])*(p2[1] - p3[1]) - (p2[0] - p3[0])*(p1[1] - p3[1])

def is_point_in_triangle_cross_product(A, B, C, P):
    d1 = sign(P, A, B)
    d2 = sign(P, B, C)
    d3 = sign(P, C, A)

    has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
    has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)

    return not (has_neg and has_pos)


## Calculate Depth and color if it is in the triangle

We calculate them through interpolation (Barycentric coordinates). All values are stored in the vertexes.

reference: [https://hackmd.io/@leon890820/rkI-2oYXi](https://hackmd.io/@leon890820/rkI-2oYXi)

### Barycentric interpolation

- input
  - position1, value1
  - position2, value2
  - position3, value3
  - x, y
- output
  - value corresponding to (x, y)

In [80]:
def Barycentric_interpolation( P1, value1, P2, value2, P3, value3, x, y):

  x1 = P1[0]
  x2 = P2[0]
  x3 = P3[0]

  y1 = P1[1]
  y2 = P2[1]
  y3 = P3[1]

  A1 = 0.5*abs((x*(y2-y3)) + (x2*(y3-y)) + (x3*(y-y2)))
  A2 = 0.5*abs((x1*(y-y3)) + (x*(y3-y1)) + (x3*(y1-y)))
  A3 = 0.5*abs((x1*(y2-y)) + (x2*(y-y1)) + (x*(y1-y2)))
  Atotal = 0.5*abs((x1*(y2-y3)) + (x2*(y3-y1)) + (x3*(y1-y2)))

  L1 = A1 / Atotal
  L2 = A2 / Atotal
  L3 = A3 / Atotal

  return L1*value1 + L2*value2 + L3*value3



In [81]:
def GetDepth(position1,position2,position3,position1_depth,position2_depth,position3_depth,x,y):
  x1 = position1[0]
  x2 = position2[0]
  x3 = position3[0]

  y1 = position1[1]
  y2 = position2[1]
  y3 = position3[1]

  A1 = 0.5*abs((x*(y2-y3)) + (x2*(y3-y)) + (x3*(y-y2)))
  A2 = 0.5*abs((x1*(y-y3)) + (x*(y3-y1)) + (x3*(y1-y)))
  A3 = 0.5*abs((x1*(y2-y)) + (x2*(y-y1)) + (x*(y1-y2)))
  Atotal = 0.5*abs((x1*(y2-y3)) + (x2*(y3-y1)) + (x3*(y1-y2)))

  L1 = A1 / Atotal
  L2 = A2 / Atotal
  L3 = A3 / Atotal

  return L1*position1_depth + L2*position2_depth + L3*position3_depth



In [82]:
def GetColor( position1, position2, position3, position1_color, position2_color, position3_color, x, y):
  x1 = position1[0]
  x2 = position2[0]
  x3 = position3[0]

  y1 = position1[1]
  y2 = position2[1]
  y3 = position3[1]

  A1 = 0.5*abs((x*(y2-y3)) + (x2*(y3-y)) + (x3*(y-y2)))
  A2 = 0.5*abs((x1*(y-y3)) + (x*(y3-y1)) + (x3*(y1-y)))
  A3 = 0.5*abs((x1*(y2-y)) + (x2*(y-y1)) + (x*(y1-y2)))
  Atotal = 0.5*abs((x1*(y2-y3)) + (x2*(y3-y1)) + (x3*(y1-y2)))

  L1 = A1 / Atotal
  L2 = A2 / Atotal
  L3 = A3 / Atotal

  #position1_color: np.array([R,G,B])
  return L1*position1_color + L2*position2_color + L3*position3_color


## Write them into screen buffer

In [92]:
from PIL import Image
mesh = Mesh()
mesh.load_ply( 'C:/Users/hcn12/Desktop/NING/Github/3D-rendering-accelerator/code/SW/mesh/bunny.ply' )

eyetmp = [5.239, 4.517, 1.538]
CamX = eyetmp[0]
CamY = eyetmp[1]
CamZ = eyetmp[2]
camera = Camera( eyeX=CamX, eyeY=CamY, eyeZ=CamZ,
centerX=0, centerY=0, centerZ=0,
upX=0, upY=1, upZ=0,
screen_W=1280, screen_H=720,
near_clipping_plane=0.1, far_clipping_plane=10,
fov=45 )

perform_MVP_transformation(camera, mesh)
NDC_transformation(mesh)
NDC_to_screen(camera, mesh)

for face in mesh.faces:

    P1 = mesh.screen_space_vertices[face[0]]
    P2 = mesh.screen_space_vertices[face[1]]
    P3 = mesh.screen_space_vertices[face[2]]
    P1_depth = mesh.NDC_vertices[face[0]][2]
    P2_depth = mesh.NDC_vertices[face[1]][2]
    P3_depth = mesh.NDC_vertices[face[2]][2]

    P1_color = mesh.colors[face[0]]
    P2_color = mesh.colors[face[1]]
    P3_color = mesh.colors[face[2]]

    for w in range( int(min( P1[0], P2[0], P3[0] ) + 0.5), int(max( P1[0], P2[0], P3[0] ) + 0.5) +1 ):
        for h in range( int(min( P1[1], P2[1], P3[1] ) + 0.5), int(max( P1[1], P2[1], P3[1] ) + 0.5) +1 ):
            if is_point_in_triangle_cross_product( P1, P2, P3, [w, h] ):
                # get depth
                depth = GetDepth( P1, P2, P3, P1_depth, P2_depth, P3_depth, w, h )
                if (depth <= camera.screen_depth_buffer[h][w]):
                    camera.screen_depth_buffer[h][w] = depth
                    camera.screen_buffer[h][w] =  GetColor( P1, P2, P3, P1_color, P2_color, P3_color, w, h )

# Show the rendered image
img = Image.fromarray(camera.screen_buffer)
img.show()

In [ ]:
from PIL import Image
box = Mesh()
box.load_ply( '/content/drive/MyDrive/ICLAB_mesh/box.ply' )

frames = []
with tqdm(total=(2 * 3 * 360) // (2 * 3), desc="Rendering frames", postfix={"Camera position": "(0, 0, 0)"}) as pbar:
    for rad in range(0, 2 * 3 * 360, 2 * 3):
        distance = 5
        rad /= 360
        CamX = distance * math.cos(rad)
        CamY = distance
        CamZ = distance * math.sin(rad)
        camera = Camera( eyeX=CamX, eyeY=CamY, eyeZ=CamZ,
          centerX=0, centerY=0, centerZ=0,
          upX=0, upY=1, upZ=0,
          screen_W=1280, screen_H=720,
          near_clipping_plane=0.1, far_clipping_plane=1000,
          fov=45 )
        # print(f"Camera position: ({camera.eyeX}, {camera.eyeY}, {camera.eyeZ})")

        # Update the progress bar's postfix with the current camera position
        pbar.set_postfix({"Camera position": f"({CamX:.2f}, {CamY:.2f}, {CamZ:.2f})"})

        perform_MVP_transformation(camera, box)
        NDC_transformation(box)
        NDC_to_screen(camera, box)

        for face in box.faces:
          # print("Face vertices indices:", face)
          P1 = box.screen_space_vertices[face[0]]
          P2 = box.screen_space_vertices[face[1]]
          P3 = box.screen_space_vertices[face[2]]
          P1_depth = box.NDC_vertices[face[0]][2]
          P2_depth = box.NDC_vertices[face[1]][2]
          P3_depth = box.NDC_vertices[face[2]][2]

          P1_color = box.colors[face[0]]
          P2_color = box.colors[face[1]]
          P3_color = box.colors[face[2]]

          for w in range( int(min( P1[0], P2[0], P3[0] ) + 0.5), int(max( P1[0], P2[0], P3[0] ) + 0.5) +1 ):
            for h in range( int(min( P1[1], P2[1], P3[1] ) + 0.5), int(max( P1[1], P2[1], P3[1] ) + 0.5) +1 ):
              if is_point_in_triangle_cross_product( P1, P2, P3, [w, h] ):
                # get depth
                depth = GetDepth( P1, P2, P3, P1_depth, P2_depth, P3_depth, w, h )
                if (depth <= camera.screen_depth_buffer[h][w]):
                  camera.screen_depth_buffer[h][w] = depth
                  camera.screen_buffer[h][w] =  GetColor( P1, P2, P3, P1_color, P2_color, P3_color, w, h )
          # print("original vertices", box.vertices[face[0]], box.vertices[face[1]], box.vertices[face[2]])
          # print("screen space vertices",P1,P2,P3)

        frames.append(camera.screen_buffer)
        # Update the progress bar
        pbar.update(1)

# Convert arrays to PIL Image objects
images = [Image.fromarray(frame, mode='RGB') for frame in frames]

# Save as a GIF
images[0].save(
    "output.gif",
    save_all=True,
    append_images=images[1:],  # Add the rest of the frames
    duration=5,             # Duration of each frame in milliseconds
    loop=0                    # 0 means infinite loop
)


In [ ]:
# Save as a GIF
images[0].save(
    "output.gif",
    save_all=True,
    append_images=images[1:],  # Add the rest of the frames
    duration=10,             # Duration of each frame in milliseconds
    loop=0                    # 0 means infinite loop
)

In [ ]:
depth_map = []
MAX = max(camera.screen_depth_buffer.flatten())
MIN = min(camera.screen_depth_buffer.flatten())

for i in range(camera.screen_depth_buffer.shape[0]):
  tmp = []
  for j in range(camera.screen_depth_buffer.shape[1]):
    tmp.append([(camera.screen_depth_buffer[i][j] - MIN) / (MAX - MIN), 255, 255])
  depth_map.append(tmp)

plt.imshow(depth_map)
plt.axis('off')
plt.show()